# 🚦 RT-DETR 交通标志检测 — Google Colab 训练

**数据集**: GTSRB (German Traffic Sign Recognition Benchmark) — 43 类

**模型**: RT-DETR-L (Ultralytics)

---
## 使用步骤
1. 菜单 → **修改 → 笔记本设置** → 硬件加速器选择 **GPU (T4 / A100)**
2. 按顺序运行每个单元格，或点击 **全部运行**
3. 训练完成后下载 `best.pt` 权重替换本地文件

## 0. 检查 GPU 环境

In [ ]:
import torch

print('PyTorch 版本:', torch.__version__)
print('CUDA 可用:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU 型号:', torch.cuda.get_device_name(0))
    print('显存大小:', f'{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️ 未检测到 GPU！请前往 修改→笔记本设置 开启 GPU 加速器')

!nvidia-smi

## 1. 挂载 Google Drive（用于保存训练结果）

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = '/content/drive/MyDrive/rtdetr_gtsrb'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'✅ 训练结果将同步保存至 Google Drive: {SAVE_DIR}')

## 2. 安装依赖

In [ ]:
!pip install -q ultralytics>=8.1.0

# 验证安装
import ultralytics
print('Ultralytics 版本:', ultralytics.__version__)
ultralytics.checks()

## 3. 准备 GTSRB 数据集

> **方式 A（推荐）**: 从 Kaggle 下载（需要上传 `kaggle.json`）  
> **方式 B**: 从官方网站下载并手动上传  
> **方式 C**: 使用已上传至 Google Drive 的数据集

请根据你的情况选择其中一种方式运行，**三种方式只需执行一种**。

In [ ]:
# ============================================================
# 方式 A: 从 Kaggle 下载（推荐，最简单）
# 前提：需要先运行 from google.colab import files; files.upload()
#       上传你的 kaggle.json API 凭据
# ============================================================

# 步骤 1: 上传 kaggle.json
from google.colab import files
print('请上传你的 kaggle.json 文件（来自 https://www.kaggle.com/account → Create New Token）')
uploaded = files.upload()  # 弹出文件选择窗口

# 步骤 2: 配置 Kaggle
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# 步骤 3: 下载数据集
!kaggle datasets download -d meowmeowmeowmeowmeow/gtsrb-german-traffic-sign -p /content/
!unzip -q /content/gtsrb-german-traffic-sign.zip -d /content/gtsrb_raw
print('✅ 方式A: 数据集下载完成')

In [ ]:
# ============================================================
# 方式 C: 从 Google Drive 加载已有数据集（如已上传过则无需重新下载）
# 将你的数据集压缩包上传到 Google Drive 后取消注释
# ============================================================

# GDRIVE_ZIP = '/content/drive/MyDrive/gtsrb_raw.zip'   # 修改为你的文件路径
# !unzip -q "{GDRIVE_ZIP}" -d /content/gtsrb_raw
# print('✅ 方式C: 从 Drive 加载完成')

## 4. 将原始数据转换为 YOLO 格式

GTSRB 原始格式为 `.ppm` 图片 + CSV 标注，需转换为 YOLO 目标检测格式（`images/` + `labels/`）。

In [ ]:
import os
import csv
import shutil
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm

RAW_DIR    = Path('/content/gtsrb_raw')
YOLO_DIR   = Path('/content/datasets/gtsrb')

def convert_gtsrb_to_yolo(raw_dir: Path, yolo_dir: Path):
    """将 GTSRB 数据集转换为 YOLO 格式"""
    for split in ['train', 'val']:
        (yolo_dir / 'images' / split).mkdir(parents=True, exist_ok=True)
        (yolo_dir / 'labels' / split).mkdir(parents=True, exist_ok=True)

    # ---- 训练集（含标注 CSV）----
    train_csv = raw_dir / 'Train.csv'
    if not train_csv.exists():
        # 尝试 Kaggle 数据集结构
        train_csv = raw_dir / 'gtsrb-german-traffic-sign' / 'Train.csv'
        raw_dir   = raw_dir / 'gtsrb-german-traffic-sign'

    print(f'读取训练标注: {train_csv}')
    rows = []
    with open(train_csv, 'r') as f:
        reader = csv.DictReader(f)
        rows = list(reader)

    # 打乱并划分 80% 训练 / 20% 验证
    import random
    random.seed(42)
    random.shuffle(rows)
    split_idx = int(len(rows) * 0.8)
    splits = {'train': rows[:split_idx], 'val': rows[split_idx:]}

    for split, split_rows in splits.items():
        print(f'转换 {split} 集: {len(split_rows)} 张')
        for row in tqdm(split_rows, desc=split):
            img_path = raw_dir / row['Path']
            if not img_path.exists():
                continue

            class_id = int(row['ClassId'])
            x1, y1, x2, y2 = int(row['Roi.X1']), int(row['Roi.Y1']), int(row['Roi.X2']), int(row['Roi.Y2'])
            w_img, h_img = int(row['Width']), int(row['Height'])

            # 转换为 YOLO 归一化格式 (cx, cy, w, h)
            cx = ((x1 + x2) / 2) / w_img
            cy = ((y1 + y2) / 2) / h_img
            bw = (x2 - x1) / w_img
            bh = (y2 - y1) / h_img

            # 保存图片（转 jpg）
            stem = f'{split}_{class_id}_{img_path.stem}'
            dst_img  = yolo_dir / 'images' / split / f'{stem}.jpg'
            dst_lbl  = yolo_dir / 'labels' / split / f'{stem}.txt'

            img = Image.open(img_path).convert('RGB')
            img.save(dst_img, 'JPEG', quality=95)

            with open(dst_lbl, 'w') as lf:
                lf.write(f'{class_id} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}\n')

    print('✅ 数据转换完成！')
    print(f'   训练集: {len(list((yolo_dir/"images"/"train").iterdir()))} 张')
    print(f'   验证集: {len(list((yolo_dir/"images"/"val").iterdir()))} 张')


convert_gtsrb_to_yolo(RAW_DIR, YOLO_DIR)

## 5. 生成数据集配置文件 gtsrb.yaml

In [ ]:
yaml_content = """
train: /content/datasets/gtsrb/images/train
val:   /content/datasets/gtsrb/images/val

nc: 43

names: [
  'speed limit 20', 'speed limit 30', 'speed limit 50', 'speed limit 60',
  'speed limit 70', 'speed limit 80', 'end of speed limit 80', 'speed limit 100',
  'speed limit 120', 'no passing', 'no passing for vehicles over 3.5 metric tons',
  'right-of-way at the next intersection', 'priority road', 'yield', 'stop',
  'no vehicles', 'vehicles over 3.5 metric tons prohibited', 'no entry',
  'general caution', 'dangerous curve to the left', 'dangerous curve to the right',
  'double curve', 'bumpy road', 'slippery road', 'road narrows on the right',
  'road work', 'traffic signals', 'pedestrians', 'children crossing',
  'bicycles crossing', 'beware of ice/snow', 'wild animals crossing',
  'end of all speed and passing limits', 'turn right ahead', 'turn left ahead',
  'ahead only', 'go straight or right', 'go straight or left', 'keep right',
  'keep left', 'roundabout mandatory', 'end of no passing',
  'end of no passing by vehicles over 3.5 metric tons'
]
"""

with open('/content/gtsrb.yaml', 'w') as f:
    f.write(yaml_content)

print('✅ gtsrb.yaml 生成完成')
!cat /content/gtsrb.yaml

## 6. 开始训练 RT-DETR

| 参数 | 建议值（T4 16GB） | 建议值（A100 40GB） |
|------|-----------------|-------------------|
| `MODEL_SIZE` | `rtdetr-l` | `rtdetr-x` |
| `BATCH_SIZE` | `8` | `16` |
| `EPOCHS` | `100` | `150` |
| `IMG_SIZE` | `640` | `640` |

> ⏱️ T4 GPU 上训练 100 epoch 约需 **4~6 小时**，请保持标签页不要关闭，或启用 `SAVE_INTERVAL` 定期保存

In [ ]:
from ultralytics import RTDETR
import torch

# ============================================================
# 训练超参数 —— 按需修改
# ============================================================
MODEL_SIZE  = 'rtdetr-l'              # rtdetr-l 或 rtdetr-x
DATA_YAML   = '/content/gtsrb.yaml'
EPOCHS      = 100
IMG_SIZE    = 640
BATCH_SIZE  = 8                        # T4: 8 | A100: 16
WORKERS     = 2                        # Colab 建议不超过 2
PROJECT     = '/content/drive/MyDrive/rtdetr_gtsrb'  # 直接保存到 Drive
NAME        = 'train_v1'
DEVICE      = 0 if torch.cuda.is_available() else 'cpu'
# ============================================================

print(f'🚀 使用设备: {DEVICE}')
print(f'📦 模型: {MODEL_SIZE}.pt')
print(f'🗂️  数据: {DATA_YAML}')
print(f'🔁 Epoch: {EPOCHS}  批大小: {BATCH_SIZE}  图像尺寸: {IMG_SIZE}')
print(f'💾 结果保存至: {PROJECT}/{NAME}')

model = RTDETR(f'{MODEL_SIZE}.pt')   # 自动下载官方预训练权重

results = model.train(
    data          = DATA_YAML,
    epochs        = EPOCHS,
    imgsz         = IMG_SIZE,
    batch         = BATCH_SIZE,
    device        = DEVICE,
    workers       = WORKERS,
    project       = PROJECT,
    name          = NAME,
    # 优化器超参数
    lr0           = 1e-4,
    lrf           = 0.01,
    warmup_epochs = 3,
    patience      = 30,              # 早停：30 epoch 无提升则停止
    # 数据增强
    hsv_h         = 0.015,
    hsv_s         = 0.7,
    hsv_v         = 0.4,
    flipud        = 0.0,             # 交通标志不应上下翻转
    fliplr        = 0.0,             # 交通标志不应左右翻转
    mosaic        = 0.5,
    # 训练行为
    save          = True,
    save_period   = 10,              # 每 10 epoch 保存一次检查点
    verbose       = True,
    plots         = True,
)

print(f'\n✅ 训练完成！')
print(f'   最佳权重: {results.save_dir}/weights/best.pt')
print(f'   最终权重: {results.save_dir}/weights/last.pt')

## 7. 训练结果评估

In [ ]:
from ultralytics import RTDETR
import os

BEST_PT = f'{PROJECT}/{NAME}/weights/best.pt'

print(f'📊 加载最佳权重进行评估: {BEST_PT}')
model_eval = RTDETR(BEST_PT)

metrics = model_eval.val(
    data    = DATA_YAML,
    imgsz   = IMG_SIZE,
    device  = DEVICE,
    verbose = True,
)

print('\n📈 验证集指标:')
print(f'  mAP@50:     {metrics.box.map50:.4f}')
print(f'  mAP@50-95:  {metrics.box.map:.4f}')
print(f'  Precision:  {metrics.box.mp:.4f}')
print(f'  Recall:     {metrics.box.mr:.4f}')

## 8. 可视化训练曲线

In [ ]:
from IPython.display import Image as IPyImage, display
import glob
import os

TRAIN_DIR = f'{PROJECT}/{NAME}'

# 显示训练曲线
curve_path = os.path.join(TRAIN_DIR, 'results.png')
if os.path.exists(curve_path):
    print('📉 训练损失 & 指标曲线:')
    display(IPyImage(curve_path, width=900))

# 显示混淆矩阵
for cm_file in ['confusion_matrix.png', 'confusion_matrix_normalized.png']:
    cm_path = os.path.join(TRAIN_DIR, cm_file)
    if os.path.exists(cm_path):
        print(f'\n🎯 {cm_file}:')
        display(IPyImage(cm_path, width=900))

# 显示验证样本
val_imgs = glob.glob(os.path.join(TRAIN_DIR, 'val_batch*.jpg'))
for img_path in val_imgs[:2]:
    print(f'\n🖼️ {os.path.basename(img_path)}:')
    display(IPyImage(img_path, width=900))

## 9. 推理测试（可选）

用训练好的模型对一张图片进行推理测试。

In [ ]:
import glob
import random
from IPython.display import Image as IPyImage, display

# 随机从验证集取一张图片测试
val_images = glob.glob('/content/datasets/gtsrb/images/val/*.jpg')
test_img   = random.choice(val_images)
print(f'测试图片: {test_img}')

results = model_eval.predict(
    test_img,
    conf    = 0.25,
    iou     = 0.45,
    save    = False,
    verbose = False,
)

result   = results[0]
annotated = result.plot()

# 保存并显示
import cv2
save_path = '/content/test_result.jpg'
cv2.imwrite(save_path, annotated)
display(IPyImage(save_path, width=640))

print(f'\n检测到 {len(result.boxes)} 个目标:')
for box in result.boxes:
    cls_id = int(box.cls[0])
    conf   = float(box.conf[0])
    name   = result.names[cls_id]
    print(f'  [{cls_id:02d}] {name:50s}  置信度: {conf:.3f}')

## 10. 下载训练好的权重

> 权重已同步保存到 **Google Drive**，可直接在 Drive 中找到。  
> 若还需在 Colab 本地直接下载，运行以下单元格。

In [ ]:
from google.colab import files
import os

BEST_PT = f'{PROJECT}/{NAME}/weights/best.pt'

if os.path.exists(BEST_PT):
    print(f'📥 正在下载: {BEST_PT}')
    files.download(BEST_PT)
    print('✅ 下载完成！将 best.pt 放到项目根目录，然后在 .env 中设置:')
    print('   WEIGHTS_PATH=best.pt')
else:
    print(f'❌ 找不到权重文件: {BEST_PT}')
    print('   请先完成训练（第 6 节）')